### Setting paths and imports

In [1]:
import sys
import os
import pandas as pd

module_path = os.path.abspath(os.path.join('..'))
sys.path.append(module_path)
module_path

os.chdir(module_path)

# from src.evaluation.evaluator_manager import EvaluatorManager
# from src.evaluation.future.evaluator_manager_do import EvaluatorManagers as PairedEvaluatorManager
from src.utils.context import Context
from src.data_analysis.future.data_analyzer import DataAnalyzer as data_analyzer

from src.evaluation.future.evaluator_manager_triplets import EvaluatorManager

config_f_name = 'bls_selection_net_trainable/bzr/dcm/dcm-lcls/generate_minimize1.jsonc'

### Evaluating the explainer

In [ ]:
config_path = os.path.join(module_path, 'lab', 'config', config_f_name)
runno = 1
    
print(f"Generating context for: {config_path}")
context = Context.get_context(config_path)
context.run_number = runno

context.logger.info(f"Executing: {context.config_file} Run: {context.run_number}")
context.logger.info("Creating the evaluation manager....................................")

context.logger.info("Creating the evaluators......................................................")
eval_manager = EvaluatorManager(context)

context.logger.info(
    "Evaluating the explainers............................................................."
)
eval_manager.evaluate()

Generating context for: c:\Users\rodri\OneDrive\Documents\Projects\GRETEL stuff\GRETEL\lab\config\bls_selection_net_trainable/bzr/dcm/dcm-lcls/generate_minimize1.jsonc
2026-01-08 19:01:18,457 | INFO | 43592 - Executing: c:\Users\rodri\OneDrive\Documents\Projects\GRETEL stuff\GRETEL\lab\config\bls_selection_net_trainable/bzr/dcm/dcm-lcls/generate_minimize1.jsonc Run: 1
2026-01-08 19:01:18,486 | INFO | 43592 - Creating the evaluation manager....................................
2026-01-08 19:01:18,517 | INFO | 43592 - Creating the evaluators......................................................
2026-01-08 19:01:18,621 | INFO | 43592 - Loading: BZR-56626c286d62df32ca4f909426d2b237
2026-01-08 19:01:18,656 | INFO | 43592 - Instantiating: src.dataset.generators.TUDataset.TUDataset
2026-01-08 19:01:18,729 | INFO | 43592 - Dataset Data Path:	./data/working/BZR
2026-01-08 19:01:18,770 | INFO | 43592 - Loaded BZR with 405 graphs.
2026-01-08 19:01:18,932 | INFO | 43592 - Dataset Data Path:	./data/

c:\Users\rodri\OneDrive\Documents\Projects\GRETEL stuff\GRETEL\src\core\factory_base.py:33: SyntaxWarning: invalid escape sequence '\('
  __cls_param_ptrn = re.compile('(^.*)'+ '\(' +'(.*)'+'\)')
c:\Users\rodri\OneDrive\Documents\Projects\GRETEL stuff\GRETEL\src\core\factory_base.py:33: SyntaxWarning: invalid escape sequence '\)'
  __cls_param_ptrn = re.compile('(^.*)'+ '\(' +'(.*)'+'\)')


KeyboardInterrupt: 

### Aggregating the stats

In [2]:
results_path = os.path.join(module_path, 'lab', 'output', 'results')
stats_file_path = os.path.join(module_path, 'lab', 'stats', 'results.csv')
res = data_analyzer.create_aggregated_dataframe(results_path)

# Order rows by dataset, then generator, then minimizer (scope format:
# '<dataset>_<gen>_<variant>'). Use rsplit so dataset slugs containing
# hyphens (e.g. 'tcr-ablation-cycles', 'colors-3') stay intact.
_parts = res['scope'].astype(str).str.rsplit('_', n=2, expand=True)
_parts.columns = ['_dataset', '_gen', '_min']
res = res.assign(**_parts).sort_values(
    ['_dataset', '_gen', '_min'], kind='stable', na_position='last',
).drop(columns=['_dataset', '_gen', '_min']).reset_index(drop=True)

os.makedirs(os.path.dirname(stats_file_path), exist_ok=True)
res.to_csv(stats_file_path, index=False)
res

,scope,dataset,oracle,explainer,Runtime,Runtime-std,GraphEditDistance,GraphEditDistance-std,FeatureEditDistance,FeatureEditDistance-std,Correctness,Correctness-std,OracleCalls,OracleCalls-std,OracleAccuracy,OracleAccuracy-std,Sparsity,Sparsity-std
0,aids_dcm_lcls-net-trainable,AIDS-196bfc784dde399b8a913e5706c8669b,OracleTorch-3d12f86f6853c575b5dbafdbedd376eb,GenerateMinimize(DCM-LocalSearch),72.511390,2.361574,11.803000,0.768600,67.468504,7.946167,1.000000,0.000000,1133.124000,38.873773,1.000000,0.000000,0.109062,0.005918
1,aids_dfs_dbs,AIDS-196bfc784dde399b8a913e5706c8669b,OracleTorch-3d12f86f6853c575b5dbafdbedd376eb,GenerateMinimize(DFS-DBS),10.101920,13.804947,482.050649,446.144307,0.000000,0.000000,0.039000,0.011576,1025.515000,36.783150,1.000000,0.000000,4.701952,4.422414
2,aids_ofs_obs,AIDS-196bfc784dde399b8a913e5706c8669b,OracleTorch-3d12f86f6853c575b5dbafdbedd376eb,GenerateMinimize(OFS-OBS),3.544209,2.459894,6.693795,0.489857,0.000000,0.000000,0.051000,0.011576,2120.652000,26.533129,1.000000,0.000000,0.063807,0.004771
3,asd_dcm_lcls,ASD-15273954d84e872cf0b021cd4477bfdc,ASDOracle-9e4f3586dc330143b7849fc540b25739,GenerateMinimize(DCM-LocalSearch),1.588704,0.520154,12.052619,4.174972,0.000000,0.000000,0.467273,0.145238,1972.706364,823.711443,0.772727,0.088233,0.027225,0.009545
4,asd_dcm_lcls-net-trainable,ASD-15273954d84e872cf0b021cd4477bfdc,ASDOracle-9e4f3586dc330143b7849fc540b25739,GenerateMinimize(DCM-LocalSearch),63.806053,13.171503,10.133333,2.677478,0.000000,0.000000,0.984848,0.033880,1332.924242,100.551615,0.754545,0.093891,0.022850,0.006090
5,asd_dfs_dbs,ASD-15273954d84e872cf0b021cd4477bfdc,ASDOracle-9e4f3586dc330143b7849fc540b25739,GenerateMinimize(DFS-DBS),1.123437,0.063583,12.217273,2.636622,0.000000,0.000000,1.000000,0.000000,4051.495455,14.806780,0.772727,0.088233,0.027543,0.005973
6,asd_ofs_obs,ASD-15273954d84e872cf0b021cd4477bfdc,ASDOracle-9e4f3586dc330143b7849fc540b25739,GenerateMinimize(OFS-OBS),0.830171,0.047644,11.296627,2.257513,0.000000,0.000000,0.772727,0.088233,4202.704545,71.354182,0.772727,0.088233,0.025477,0.005138
7,bbbp_dcm_lcls-net-trainable,BBBP-c0cc58bdf4acf93936322304bf6b669e,OracleTorch-67fdf45735a1c8aaece5ab786ca3c38e,GenerateMinimize(DCM-LocalSearch),35.605535,2.972382,3.326157,0.190895,60.047040,3.471724,1.000000,0.000000,515.818041,24.179882,0.899445,0.021041,0.023016,0.001355
8,bbbp_dfs_dbs,BBBP-c0cc58bdf4acf93936322304bf6b669e,OracleTorch-67fdf45735a1c8aaece5ab786ca3c38e,GenerateMinimize(DFS-DBS),18.093569,5.416713,4.548559,1.431772,0.000000,0.000000,0.603922,0.023076,3121.925490,53.387161,0.896078,0.012166,0.031417,0.009617
9,bbbp_ofs_obs,BBBP-c0cc58bdf4acf93936322304bf6b669e,OracleTorch-67fdf45735a1c8aaece5ab786ca3c38e,GenerateMinimize(OFS-OBS),6.075657,0.774421,2.816887,0.162829,0.000000,0.000000,0.784314,0.010282,3817.929412,30.107856,0.897059,0.025753,0.019339,0.001193
